In [1]:
import pandas as pd
import numpy as np
import time

import xgboost as xgb
import lightgbm as lgb

from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df = pd.read_parquet("../Dataset/final-dataset/dataset-with-features.parquet")
df_model = df.dropna(subset=["target_log_flux_1h"]).copy()

In [2]:
train_df = df_model.loc["2011":"2018"]
test_df = df_model.loc["2019"]

In [3]:
TARGET = "target_log_flux_1h"

ignore_columns = ["E2W_COR_FLUX", TARGET]
feature_columns = [col for col in train_df.columns if col not in ignore_columns]

X_train = train_df[feature_columns]
Y_train_log = train_df[TARGET]

X_test = test_df[feature_columns]
Y_test_log = test_df[TARGET]

Y_test_raw = (10 ** Y_test_log) - 1


In [4]:

print(f"Training Set (2011–2018):  {len(X_train):,} rows")
print(f"Testing Set  (2019):       {len(X_test):,} rows")
print(f"No. of Input Features:     {len(feature_columns)}")

Training Set (2011–2018):  3,308,735 rows
Testing Set  (2019):       364,469 rows
No. of Input Features:     37


In [5]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

models = {
    "XGBoost": xgb_model,
    "LightGBM": lgb_model
}

In [6]:
results = []

print("\nModel Training and Evaluation Started.")

for name, model in models.items():
    print(f"\nTraining {name}")

    # Training and logging
    start = time.time()
    model.fit(X_train, Y_train_log)
    train_time = time.time() - start
    
    # Testing
    preds_log = model.predict(X_test)
    
    # Converting log predictions to raw scale (pfu)
    preds_raw = np.maximum(0, (10 ** preds_log) - 1)
    
    # log scale evaluation
    log_r2   = r2_score(Y_test_log, preds_log) * 100
    log_rmse = np.sqrt(mean_squared_error(Y_test_log, preds_log))
    log_mae  = mean_absolute_error(Y_test_log, preds_log)
    
    # raw scale evaluation
    raw_r2   = r2_score(Y_test_raw, preds_raw) * 100
    raw_rmse = np.sqrt(mean_squared_error(Y_test_raw, preds_raw))
    raw_mae  = mean_absolute_error(Y_test_raw, preds_raw)
    
    results.append({
        "Model Name": name,
        "Training Time (s)": round(train_time, 2),
        "Log R² (%)": round(log_r2, 2),
        "Log RMSE": round(log_rmse, 4),
        "Log MAE": round(log_mae, 4),
        "Raw R² (%)": round(raw_r2, 2),
        "Raw RMSE (pfu)": round(raw_rmse, 4),
        "Raw MAE (pfu)": round(raw_mae, 4)
    })



Model Training and Evaluation Started.

Training XGBoost

Training LightGBM


In [7]:

leaderboard = pd.DataFrame(results)

print("\nEvaluation Results:\n")
print(leaderboard.to_string(index=False))


Evaluation Results:

Model Name  Training Time (s)  Log R² (%)  Log RMSE  Log MAE  Raw R² (%)  Raw RMSE (pfu)  Raw MAE (pfu)
   XGBoost              25.38       96.61    0.1441   0.0956       91.65       3684.4393       750.1664
  LightGBM              12.95       96.55    0.1455   0.0970       91.55       3707.5307       765.3756
